# 🔍 Notebook 01 — Exploratory Data Analysis & Preprocessing
> **Purpose:** Understand LOBSTER data structure, visualise LOB dynamics,
> identify data quality issues, and produce the cleaned dataset.

**Inputs:** `data/lobster_clean.parquet`  
**Outputs:** `data/lobster_features.parquet`

---

## 1.1  Load cleaned data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline

df = pd.read_parquet('data/lobster_clean.parquet')
print(f'Shape: {df.shape}')
df.head(3)

## 1.2  Dataset overview

In [ ]:
from utils.lobster_loader import describe_dataset, MSG_TYPE
from utils.feature_builder import add_mid_and_spread

df = add_mid_and_spread(df)
describe_dataset(df)

## 1.3  Mid-price time series

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Mid price
axes[0].plot(df.index, df['mid_price'], lw=0.6, color='steelblue')
axes[0].set_ylabel('Mid Price ($)')
axes[0].set_title('LOBSTER Mid Price — Full Session')

# Spread
axes[1].fill_between(df.index, df['spread_bps'], alpha=0.5, color='darkorange')
axes[1].set_ylabel('Spread (bps)')
axes[1].set_xlabel('Event Index')
axes[1].set_title('Bid-Ask Spread (bps)')

plt.tight_layout()
plt.savefig('data/fig_mid_spread.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.4  Event type distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Count
type_counts = df['Type'].value_counts().sort_index()
labels = [f'Type {t}\n{MSG_TYPE.get(t,"?")}' for t in type_counts.index]
axes[0].bar(labels, type_counts.values, color=sns.color_palette('muted', len(labels)))
axes[0].set_title('Message Event Type Distribution')
axes[0].set_ylabel('Count')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=8)

# Direction
dir_counts = df['Direction'].value_counts()
axes[1].bar(['Sell (-1)', 'Buy (+1)'],
            [dir_counts.get(-1,0), dir_counts.get(1,0)],
            color=['tomato', 'seagreen'])
axes[1].set_title('Order Direction')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('data/fig_event_types.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.5  LOB depth visualisation (snapshot at event 5000)

In [ ]:
snapshot_idx = 5000
snap = df.iloc[snapshot_idx]

bid_prices = [snap[f'BidP{i}'] / 10000 for i in range(1, 11)]
bid_sizes  = [snap[f'BidS{i}'] for i in range(1, 11)]
ask_prices = [snap[f'AskP{i}'] / 10000 for i in range(1, 11)]
ask_sizes  = [snap[f'AskS{i}'] for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(bid_prices, [-s for s in bid_sizes], height=0.00005,
        color='seagreen', alpha=0.8, label='Bid')
ax.barh(ask_prices, ask_sizes, height=0.00005,
        color='tomato', alpha=0.8, label='Ask')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Volume (shares)')
ax.set_ylabel('Price ($)')
ax.set_title(f'LOB Depth Profile — Event {snapshot_idx}')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_lob_depth.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.6  Order imbalance & spread distributions

In [ ]:
from utils.feature_builder import add_micro_features

df = add_micro_features(df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['imbalance_l1'].dropna(), bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Order Imbalance (Level 1)')
axes[0].set_xlabel('OBI')

axes[1].hist(df['spread_bps'].dropna().clip(0, 20), bins=80,
             color='darkorange', edgecolor='none')
axes[1].set_title('Spread (bps)')
axes[1].set_xlabel('bps')

axes[2].hist(df['depth_ratio'].dropna().clip(0, 5), bins=80,
             color='purple', edgecolor='none')
axes[2].set_title('Bid/Ask Depth Ratio')
axes[2].set_xlabel('ratio')

plt.tight_layout()
plt.savefig('data/fig_feature_dists.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.7  Label distribution (3-class target)

In [ ]:
from utils.feature_builder import make_labels

labels = make_labels(df, horizon=10, threshold_bps=0.5)
df['label'] = labels

# Drop NaNs from look-ahead window
df = df.dropna(subset=['label'])

counts = df['label'].value_counts().sort_index()
colors = ['tomato', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['DOWN (-1)', 'FLAT (0)', 'UP (+1)'], counts.values, color=colors)
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
ax.set_title('Label Distribution (horizon=10 events, ±0.5 bps threshold)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('data/fig_label_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.8  Correlation heatmap (engineered features)

In [ ]:
eng_cols = ['imbalance_l1','imbalance_l5','imbalance_l10',
            'spread_bps','depth_bid','depth_ask','depth_ratio',
            'price_range','vwap_imbalance','label']

corr = df[eng_cols].astype(float).corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.4)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('data/fig_corr.png', dpi=120, bbox_inches='tight')
plt.show()

## 1.9  Save feature-enriched dataset

In [ ]:
df.to_parquet('data/lobster_features.parquet', index=False)
print(f'Saved data/lobster_features.parquet  ({len(df):,} rows, {df.shape[1]} cols)')

---
> ✅ **EDA complete.** Proceed to `02_feature_engineering.ipynb`.